In [1]:
import os

In [3]:
datahub_server = "<INSERT_YOUR_DATAHUB_SERVER>"

In [5]:
from datahub_metadata_fetcher import (
    DatahubMetadataFetcher,
    get_all_tables_info,
    get_columns_for_table,
    search_tables,
    get_complete_table_metadata,
)


fetcher = DatahubMetadataFetcher(gms_server=datahub_server)

In [6]:
tables_df = get_all_tables_info(fetcher)

In [14]:
tables_df["table_description"].value_counts()

table_description
                                                                               33
Activity data triggered when a contact is qualified as a prospect by an SDR     2
Activity data triggered when a client is onboarded                              2
Activity data triggered when a contact is sent an email by an SDR               1
Activity data triggered when a contact unsubscribes from an email               1
                                                                               ..
Activity data triggered when a company offboards an SDR                         1
Activity data triggered when a company offboards an AE                          1
Activity data triggered when a company hires an SDR                             1
Activity data triggered when an AE accrues a quota contribution                 1
Activity data triggered when a trial ends                                       1
Name: count, Length: 81, dtype: int64

In [15]:
sample_table = tables_df.iloc[0]["table_name"]
print(f"테이블 '{sample_table}'의 컬럼 정보:")
columns_df = get_columns_for_table(fetcher, sample_table)

테이블 'client_stream_activated_on_product'의 컬럼 정보:


In [18]:
columns_df.column_description.values

array(['The primary key for this table', 'The entity id of the client',
       'The timestamp of the activity', 'The name of the activity',
       'The revenue impact of the activity',
       "JSON string containing feature data related to the activity, including customer segments such as 'active_users', \n'churn_risk_users', 'churned_users', 'free_users', 'paid_users', 'grace_period_users', 'canceled_users', \n'new_users', 'returning_users', 'trial_users' and plan types like 'basic_plan', 'standard_plan', 'premium_plan', \n'monthly_plan', 'annual_plan', 'lifetime_plan'. and csm names like 'chris', 'john', 'jane', 'jim', 'jill', 'james' \nand mrr_tier types like 'tier1', 'tier2', 'tier3', 'tier4', 'tier5'\n"],
      dtype=object)

In [21]:
tables_df.iloc[0].table_description

'Activity data triggered when a client activates on a product'

In [20]:
from IPython.display import Markdown

Markdown(columns_df.to_markdown())

|    | column_name    | column_description                                                                                                  |
|---:|:---------------|:--------------------------------------------------------------------------------------------------------------------|
|  0 | id             | The primary key for this table                                                                                      |
|  1 | entity_id      | The entity id of the client                                                                                         |
|  2 | activity_ts    | The timestamp of the activity                                                                                       |
|  3 | activity       | The name of the activity                                                                                            |
|  4 | revenue_impact | The revenue impact of the activity                                                                                  |
|  5 | feature_json   | JSON string containing feature data related to the activity, including customer segments such as 'active_users',    |
|    |                | 'churn_risk_users', 'churned_users', 'free_users', 'paid_users', 'grace_period_users', 'canceled_users',            |
|    |                | 'new_users', 'returning_users', 'trial_users' and plan types like 'basic_plan', 'standard_plan', 'premium_plan',    |
|    |                | 'monthly_plan', 'annual_plan', 'lifetime_plan'. and csm names like 'chris', 'john', 'jane', 'jim', 'jill', 'james'  |
|    |                | and mrr_tier types like 'tier1', 'tier2', 'tier3', 'tier4', 'tier5'                                                 |

In [23]:
# Inmamory embedding으로 사용자 질문에 적합한 테이블 찾기

# 사용자 질문 refine 예를 들면 동의어로 변환하거나 더 정확한 질문으로 변환

from langchain_openai.chat_models import ChatOpenAI

OPENAI_API_KEY = "<INSERT_YOUR_OPENAI_API_KEY>"

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, openai_api_key=OPENAI_API_KEY)

In [25]:
from pydantic import BaseModel
from typing import List
from langchain_core.prompts import ChatPromptTemplate


class Persona(BaseModel):
    name: str
    department: str
    role: str
    background: str


class PersonaList(BaseModel):
    personas: List[Persona]


system_prompt = """주어진 Tabel description들을 참고하여 Text2SQL 서비스로 질문을 할만한 패르소나를 생성하세요"""


user_prompt = """{tabke_name} : {table_description}"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("user", "{input}"),
    ]
)

chain = prompt | llm.with_structured_output(PersonaList)

In [42]:
# drop empty string
drop_empty_tables = tables_df[tables_df["table_description"].apply(lambda x: x != "")]
drop_empty_tables[["table_name", "table_description"]]

,table_name,table_description
0,client_stream_activated_on_product,Activity data triggered when a client activate...
1,client_stream_active_on_subscription,Activity data triggered when a customer is act...
2,client_stream_called_support,Activity data triggered when a client is calle...
3,client_stream_churned_on_product,Activity data triggered when a client churns o...
4,client_stream_closed_support_ticket,Activity data triggered when a client closes a...
...,...,...
77,deal_stream_updated_deal_amount,Activity data triggered when a deal amount is ...
78,deal_stream_updated_win_probability,Activity data triggered when a win probability...
79,deal_stream_won_opportunity,Activity data triggered when an opportunity is...
80,ga_cube_churned_revenue,Flattened OLAP cube model for ChurnedMRR


In [49]:
def get_table_des_string(tables_df):
    return_string = ""
    for index, row in tables_df.iterrows():
        return_string += f"{row['table_name']} : {row['table_description']}\n---\n"
    return return_string


description_string = get_table_des_string(drop_empty_tables)

In [51]:
response = chain.invoke({"input": description_string})

In [54]:
def pretty_print_persona(persona):
    return f"""
    Name: {persona.name}
    Department: {persona.department}
    Role: {persona.role}
    Background: {persona.background}
    """


from IPython.display import display

for persona in response.personas:
    display(Markdown(pretty_print_persona(persona)))
    print("-" * 100)


    Name: Alice Johnson
    Department: Customer Success
    Role: Customer Success Manager
    Background: Alice has over 5 years of experience in customer success and is responsible for ensuring clients are satisfied with their subscriptions. She often analyzes customer activity data to identify churn risks and opportunities for upselling.
    

----------------------------------------------------------------------------------------------------



    Name: Bob Smith
    Department: Sales
    Role: Sales Development Representative
    Background: Bob is a recent graduate with a degree in marketing. He is focused on generating leads and qualifying prospects. He frequently uses activity data to track engagement and follow up with potential customers.
    

----------------------------------------------------------------------------------------------------



    Name: Charlie Brown
    Department: Support
    Role: Support Specialist
    Background: Charlie has a background in technical support and is responsible for resolving customer issues. He uses activity data to monitor support ticket trends and improve response times.
    

...

----------------------------------------------------------------------------------------------------



    Name: Ethan Hunt
    Department: Product Management
    Role: Product Manager
    Background: Ethan has a background in software development and product management. He analyzes customer activity data to inform product improvements and feature development.
    

----------------------------------------------------------------------------------------------------



    Name: Fiona Green
    Department: Finance
    Role: Financial Analyst
    Background: Fiona has a background in finance and is responsible for analyzing revenue data. She uses activity data to forecast churned revenue and committed revenue.
    

----------------------------------------------------------------------------------------------------


In [72]:
def get_persona_prompt(persona):
    return f"""
    Name: {persona.name}
    Department: {persona.department}
    Role: {persona.role}
    Background: {persona.background}
    """


system_prompt = """당신은 <persona> 에 해당하는 사람이며 Text2SQL 서비스를 사용하고 있다. 궁금한 질문들을 아래 <format> 에 해당하는 형식으로 질문하라 질문은 다양하게 생성하라

<persona>
{persona_prompt}
</persona>

<format>
- 질문 1
- 질문 2
- 질문 3
...
- 질문 n
</format>
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
    ]
)

chain = prompt | llm

In [74]:
def split_question(question):
    question = question.content
    # remove -
    question = question.replace("- ", "")
    return question.split("\n")


def gen_question(persona):
    result = {}

    question = chain.invoke({"persona_prompt": get_persona_prompt(persona)})
    result["questions"] = split_question(question)
    result["questions_md"] = question.content
    result["persona"] = persona
    return result


result = gen_question(response.personas[0])

In [82]:
import json


def save_result(result, idx):
    # result['persona']를 JSON 직렬화 가능하도록 변환
    result["persona"] = (
        result["persona"].model_dump()
        if hasattr(result["persona"], "model_dump")
        else result["persona"].__dict__
    )

    with open(f"question_result_{idx}.json", "a", encoding="utf-8") as f:
        f.write(json.dumps(result, ensure_ascii=False))


for idx, persona in enumerate(response.personas):
    result = gen_question(persona)
    save_result(result, idx)

In [85]:
def load_result(idx):
    with open(f"question_result_{idx}.json", "r", encoding="utf-8") as f:
        return json.load(f)


questions = []

for idx in range(6):
    questions.extend(load_result(idx)["questions"])

In [91]:
from difflib import SequenceMatcher
from typing import List


def remove_duplicate_questions(
    questions: List[str], similarity_threshold: float = 0.8
) -> List[str]:
    """
    문자열 유사도를 기반으로 중복된 질문을 제거하는 함수

    Args:
        questions: 질문 목록
        similarity_threshold: 유사도 임계값 (0.0 ~ 1.0, 높을수록 더 유사해야 중복으로 간주)

    Returns:
        중복이 제거된 질문 목록
    """
    if not questions:
        return []

    # 결과를 저장할 리스트
    unique_questions = [questions[0]]

    # 각 질문에 대해 이미 추가된 질문들과 비교
    for question in questions[1:]:
        is_duplicate = False

        for unique_question in unique_questions:
            # 시퀀스 매처를 사용하여 두 문자열의 유사도 계산
            similarity = SequenceMatcher(
                None, question.lower(), unique_question.lower()
            ).ratio()

            # 유사도가 임계값을 초과하면 중복으로 간주
            if similarity >= similarity_threshold:
                is_duplicate = True
                break

        # 중복이 아니면 결과 리스트에 추가
        if not is_duplicate:
            unique_questions.append(question)

    return unique_questions


questions = remove_duplicate_questions(questions)

In [93]:
len(questions)

60

In [76]:
response.personas

[Persona(name='Alice Johnson', department='Customer Success', role='Customer Success Manager', background='Alice has over 5 years of experience in customer success and is responsible for ensuring clients are satisfied with their subscriptions. She often analyzes customer activity data to identify churn risks and opportunities for upselling.'),
 Persona(name='Bob Smith', department='Sales', role='Sales Development Representative', background='Bob is a recent graduate with a degree in marketing. He is focused on generating leads and qualifying prospects. He frequently uses activity data to track engagement and follow up with potential customers.'),
 Persona(name='Charlie Brown', department='Support', role='Support Specialist', background='Charlie has a background in technical support and is responsible for resolving customer issues. He uses activity data to monitor support ticket trends and improve response times.'),
 Persona(name='Diana Prince', department='Marketing', role='Marketing A

In [79]:
import json


def save_personas(personas):
    """
    save as jsonl file
    """
    with open("personas.jsonl", "w") as f:
        for persona in personas:
            f.write(json.dumps(persona.model_dump()) + "\n")


save_personas(response.personas)